# មេរៀនទី ០៩ - គំរូការរចនាមេតាវិញ្ញាសា


## ការតំឡើង

សៀវភៅកំណត់ត្រានេះបង្ហាញពីគំនិតរចនាប័ទ្ម Metacognition ដោយប្រើរចនាសម្ព័ន្ធ Microsoft Agent។

**កម្រិតដែលត្រូវបានត្រៀមរួច:**
- ការតំឡើង Azure OpenAI ត្រូវបានកំណត់តាមរយៈអថេរបរិស្ថាន
- Azure CLI បានបញ្ចាក់អត្តសញ្ញាណ (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## តើ Metacognition ជាអ្វី?

Metacognition គឺជា **ការគិតអំពីការគិត**។ ក្នុងបរិបទនៃភ្នាក់ងារពិសោធន៍ AI វាមានន័យថាស្ថាបនាភ្នាក់ងារដែលអាច:

- **ចំហាយខ្លួនឯង** លើផលបញ្ចេញនិងដំណើរការជំនឿរបស់ខ្លួនផ្ទាល់
- **រកឃើញកំហុស** និងស្ដារឡើងវិញដោយយកចិត្ដទុកដាក់ជាជម្រើស មិនមែនបរាជ័យដោយស្ងាត់ស្ងៀមទេ
- **វាយតម្លៃ** ថាតើយ៉ាងណាទំហំពួកវាឆ្លើយតបគឺស្មុគស្មាញនិងមានប្រយោជន៍
- **ប្តូរប្រែ** អยุทธវិធីរបស់ខ្លួន នៅពេលដែលវិធីដំបូងមិនបានលទ្ធផល (ឧទាហរណ៍៖ ត្រលប់ទៅប្រព័ន្ធជំនួយមួយ)

ភ្នាក់ងារដែលមាន metacognitive មិនត្រឹមតែឆ្លើយសំណួរប៉ុណ្ណោះ — វាសម្លឹងមើលការអនុវត្តលើខ្លួនឯង ហើយកែតម្រូវនៅពេលវេលា.


## ឧបករណ៍មុខរបរ និង វិភាគបម្រើបមក

គំរូមេតាការចេតនា​ធម្មតាមួយគឺជា **យុទ្ធសាស្រ្តបម្រុង**។ អ្នកតំណាង முயាយប្រើឧបករណ៍មុខរបរជាពិសេសជាមុន; ប្រសិនបើវាបរាជ័យ (ឧ. ខុសត្រូវ 404) អ្នកតំណាងទទួលស្គាល់ការបរាជ័យនោះ ហើយប្ដូរយ៉ាងត្រចៀកទៅឧបករណ៍បម្រុង។

វានេះស្រដៀងនឹងប្រព័ន្ធនៅពិភពលោកដែលសេវាកម្មមុខរបរ​អាចមិនមានអាច​និងអ្នកតំណាងត្រូវតែ​វិភាគខ្លួនឯងអំពីបញ្ហាដើម្បីជ្រើសរើសផ្លូវជំនួស។

ខាងក្រោម យើងកំណត់ឧបករណ៍ស្វែងរកជើងហោះហើរចំនួនពីរ៖
- **មុខរបរ** — គ្របដណ្ដប់ទីក្រុងប៉ារីស ប៉េកិង និងបាស៊ីលោកណា
- **បម្រុង** — គ្របដណ្ដប់ទីក្រុងប៊ឺរឡင် ស៊ីដនេ និងទីក្រុងញូវយ៉ក


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ផ្នែកតំណាងដែលពិចារណារួមខ្លួនជាមួយការស្ដារឡើងវិញកំហុស

ផ្នែកតំណាងខាងក្រោមត្រូវបានណែនាំឲ្យព្យាយាមប្រព័ន្ធហោះបរ​មេ πρώើមជាមុនសិន ស្គាល់កើតបញ្ហា និងបញ្ចេញទៅប្រព័ន្ធបម្រើជំនួសដោយភាពត្រៀមខ្លួន។ បន្ទាប់ពីការឆ្លើយតបនីមួយៗ វាប្រៀបធៀបទៅវិញវិញថាតើវាបានឆ្លើយបានពេញលេញចំពោះសំណួររបស់អ្នកប្រើរួចហើយឬនៅ។


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## ឧបករណ៍វាយតម្លៃខ្លួនឯង

មុខមួយទៀតនៃការយល់ដឹងពីការគិតពីខ្លួនឯងគឺ **ការវាយតម្លៃខ្លួនឯង**៖ អ្នកតំណាងមួយផ្សេងទៀត (ឬអ្នកតំណាងដូចគ្នាក្នុងការឆ្លងកាត់លើកទីពីរ) កំពុងពិនិត្យមើលចម្លើយសម្រាប់ភាពបញ្ចប់ ការត្រឹមត្រូវ និងភាពជួយសម្រួល។

ខាងក្រោមយើងបង្កើតអ្នកតំណាង `ResponseEvaluator` ដែលវាយតម្លៃចម្លើយរបស់តំណាងធ្វើដំណើរនៅលើបីទំហំ។


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## សង្ខេប

ក្នុងមេរៀននេះអ្នកបានរៀនពីរបៀបបង្កើត **ភាសីទាំងឡាយដៃគូរ** ដោយប្រើ Microsoft Agent Framework៖

- **ការឆ្លុះបញ្ចាំងខ្លួនឯង** ៖ ភាសីដែលត្រួតពិនិត្យការរំពឹងទុករបស់ខ្លួន ហើយផ្តល់ការប្រាស្រ័យទាក់ទងយ៉ាងច្បាស់ថាអ្វីកើតឡើង។
- **ការស្ដារឡើងវិញដោយមានការជំនួស** ៖ លំនាំឧបករណ៍សំខាន់ + ជំនួស ដែលភាសីចាប់សំគាល់កំហុស (ឧ. កំហុស 404) ហើយព្យាយាមប្រភពជំនួសដោយស្វ័យប្រវត្តិ។
- **ការវាយតម្លៃខ្លួនឯង** ៖ ភាសីអ្នកវាយតម្លៃបន្ថែមមួយដែលវាយតម្លៃចម្លើយសម្រាប់ភាពពេញលេញ ភាពត្រឹមត្រូវ និងភាពជួយបាន។

លំនាំទាំងនេះធ្វើឲ្យភាសីកាន់តែរឹងមាំ ភាពបញ្ញើម និងគួរឱ្យទុកចិត្ត—គុណលក្ខណៈសំខាន់សម្រាប់ការតភ្ជាប់ផលិតកម្ម។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
